In [ ]:
"""
GDELT Macro News Fetcher (Sentiment Layer)

Fetches macro-relevant news for sentiment analysis using grouped keyword queries
to minimise API calls. Implements retry/backoff handling, deduplication, source
filtering, and outputs a cleaned English-only dataset to CSV.
"""

import os
import time
import pandas as pd
from gdeltdoc import GdeltDoc, Filters
from gdeltdoc.errors import RateLimitError, ClientRequestError, ServerError

os.makedirs("../data/raw", exist_ok=True)
OUTPUT = "../data/raw/news.csv"

KEYWORD_GROUPS = [
    # ── CALM ─────────────────────────────────────────────────────────
    ["GDP growth", "economic expansion"],
    ["strong earnings", "consumer confidence"],
    ["soft landing", "rate pause"],

    # ── STRESS ───────────────────────────────────────────────────────
    ["recession", "economic slowdown"],
    ["banking crisis", "financial instability"],
    ["credit stress", "liquidity crunch"],
    ["market selloff", "volatility spike"],
    ["stock market crash", "equity selloff"],
    ["market correction", "bear market"],
    ["margin call", "forced selling"],
    ["bank run", "contagion"],
    ["debt default", "credit crunch"],
    ["inflation shock", "stagflation"],
    ["flash crash", "circuit breaker"],

    # ── NEUTRAL ──────────────────────────────────────────────────────
    ["federal reserve", "central bank"],
    ["inflation", "interest rates"],
    ["tariff", "trade war"],
]

TRUSTED_SOURCES = [ 
    'bloomberg', 'cnbc', 'marketwatch', 'morningstar', 'investing', 'ft',
    'fxstreet', 'forexlive',
    'bbc', 'reuters', 'apnews', 'nytimes', 'theguardian', 'cnn', 'euronews',
    'imf', 'worldbank', 'ecb.europa', 'federalreserve', 'oxan',
    'finance.yahoo', 'yahoo', 'businessinsider', 'forbes', 'fortune', 'bbva'
]

START_YEAR = 2019
END_YEAR   = 2026
BASE_SLEEP = 2.5

gd = GdeltDoc()


def month_range(year: int, month: int) -> tuple[str, str]:
    start = f"{year}-{month:02d}-01"
    end = f"{year+1}-01-01" if month == 12 else f"{year}-{month+1:02d}-01"
    return start, end


def safe_article_search(filters: Filters, max_retries: int = 8) -> pd.DataFrame:
    sleep_s = BASE_SLEEP

    for attempt in range(1, max_retries + 1):
        try:
            df = gd.article_search(filters)
            time.sleep(BASE_SLEEP)
            return df if df is not None else pd.DataFrame()

        except RateLimitError:
            time.sleep(sleep_s)
            sleep_s = min(sleep_s * 2.0, 120)

        except (ClientRequestError, ServerError):
            time.sleep(sleep_s)
            sleep_s = min(sleep_s * 1.5, 60)

        except Exception:
            break

    return pd.DataFrame()


all_dfs = []

print("\nStarting macro news pull...\n")

for year in range(START_YEAR, END_YEAR):
    for month in range(1, 13):
        start_date, end_date = month_range(year, month)
        month_count = 0

        for i, kw_group in enumerate(KEYWORD_GROUPS, start=1):
            f = Filters(
                keyword=kw_group,
                start_date=start_date,
                end_date=end_date,
                num_records=250
            )

            df = safe_article_search(f)

            if not df.empty:
                df["kw_group"] = i
                all_dfs.append(df)
                month_count += len(df)

        print(f"{year}-{month:02d} | Articles (raw): {month_count:,}")

if not all_dfs:
    raise RuntimeError("No articles fetched.")

news = pd.concat(all_dfs, ignore_index=True)

keep = ["url", "title", "seendate", "domain", "language", "sourcecountry", "kw_group"]
for c in keep:
    if c not in news.columns:
        news[c] = None

news = news[keep]

news.drop_duplicates(subset="url", inplace=True)
news = news[news['language'] == 'English']

pattern = '|'.join(TRUSTED_SOURCES)
news = news[
    news['url']
    .str.lower()
    .str.contains(pattern, na=False)
]

news["seendate"] = pd.to_datetime(news["seendate"], errors="coerce")
news.sort_values("seendate", inplace=True)

news.to_csv(OUTPUT, index=False)

print("\nSaved:", OUTPUT)
print("Total unique articles:", len(news))

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/news.csv")

# Filter 2024-2025 + English only
df['seendate'] = pd.to_datetime(df['seendate'])
df_filtered = df[(df['seendate'].dt.year.isin([2019, 2026]))]

# Print ALL domains
print(df_filtered['domain'].value_counts().to_string())

In [ ]:
"""
News Filtering & Validation (Sentiment Preprocessing)

Loads raw GDELT news, filters to English and credible sources,
produces monthly diagnostics, and saves a cleaned dataset
for downstream FinBERT sentiment analysis.
"""

import os
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------
RAW_PATH = "../data/raw/news.csv"
OUT_DIR  = "../data/processed/sentiment_analysis"
OUT_PATH = f"{OUT_DIR}/news_filtered.csv"

os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------------------------------------------------
# Load
# -------------------------------------------------------------------
df = pd.read_csv(RAW_PATH)

print(f"Loaded raw articles: {len(df):,}")

# -------------------------------------------------------------------
# Filters
# -------------------------------------------------------------------
CREDIBLE_SOURCES = [
    'bloomberg', 'cnbc', 'marketwatch', 'morningstar', 'investing', 'ft',
    'fxstreet', 'forexlive',
    'bbc', 'reuters', 'apnews', 'nytimes', 'theguardian', 'cnn', 'euronews',
    'imf', 'worldbank', 'ecb.europa', 'federalreserve', 'oxan',
    'finance.yahoo', 'yahoo', 'businessinsider', 'forbes', 'fortune', 'bbva'
]

# English only
df = df[df["language"] == "English"]

# Credible sources (partial domain match)
pattern = "|".join(CREDIBLE_SOURCES)
df = df[df["domain"].str.contains(pattern, case=False, na=False)]

print(f"After filtering: {len(df):,}")

# -------------------------------------------------------------------
# Dates
# -------------------------------------------------------------------
df = df.copy()
df["date"] = pd.to_datetime(df["seendate"], errors="coerce")
df.sort_values("date", inplace=True)

# -------------------------------------------------------------------
# Diagnostics
# -------------------------------------------------------------------
monthly = df.set_index("date").resample("ME").size()

print("\nMonthly article counts:")
print(monthly.to_string())

monthly.plot(title="Monthly Articles After Filter")
plt.tight_layout()
plt.show()

print("\nTop sources:")
print(df["domain"].value_counts().head(15))

# -------------------------------------------------------------------
# Save processed dataset
# -------------------------------------------------------------------
df.to_csv(OUT_PATH, index=False)

print("\nSaved filtered dataset:")
print(OUT_PATH)
print(f"Final article count: {len(df):,}")

In [ ]:
"""
News Text Inspection & Field Selection Preparation

This module loads the filtered news dataset and performs an initial
inspection of available text fields. The goal is to understand text
coverage, structure, and quality before deciding which fields will be
used for sentiment analysis.
"""

from __future__ import annotations

import pandas as pd
from typing import List


INPUT_PATH = "../data/processed/sentiment_analysis/news_filtered.csv"


# Load data
df = pd.read_csv(INPUT_PATH)

print(f"Loaded articles: {len(df):,}")


# Identify candidate text columns
TEXT_COLUMNS: List[str] = [
    col for col in df.columns
    if col.lower() in {"title", "description", "summary", "content"}
]

print("\nDetected text-related columns:")
for col in TEXT_COLUMNS:
    print(f" - {col}")

# Basic coverage diagnostics
coverage = (
    df[TEXT_COLUMNS]
    .notna()
    .mean()
    .sort_values(ascending=False)
)

print("\nNon-null coverage by column:")
print(coverage.to_string(float_format="%.2f"))

# Preview sample text
print("\nSample rows (text preview):\n")

sample = df[TEXT_COLUMNS].dropna(how="all").sample(5, random_state=42)

for i, row in sample.iterrows():
    print("-" * 80)
    for col in TEXT_COLUMNS:
        if pd.notna(row.get(col)):
            text = str(row[col]).strip()
            print(f"{col.upper()}:\n{text[:300]}{'...' if len(text) > 300 else ''}\n")

In [ ]:
"""
News Text Cleaning

Applies light, model-agnostic cleaning to news titles in preparation
for exploratory analysis and downstream sentiment modeling.
"""

from __future__ import annotations

import re
import pandas as pd

# -------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------
INPUT_PATH = "../data/processed/sentiment_analysis/news_filtered.csv"
OUTPUT_PATH = "../data/processed/sentiment_analysis/news_text_cleaned.csv"

# -------------------------------------------------------------------
# Load data
# -------------------------------------------------------------------
df = pd.read_csv(INPUT_PATH)

assert "title" in df.columns, "Expected column 'title' not found."

print(f"Loaded articles: {len(df):,}")

# -------------------------------------------------------------------
# Text cleaning utilities
# -------------------------------------------------------------------
URL_PATTERN = re.compile(r"http\S+|www\.\S+")
WHITESPACE_PATTERN = re.compile(r"\s+")


def clean_text(text: str) -> str:
    """
    Apply minimal, non-destructive cleaning to news text.

    Steps:
    - Lowercase
    - Remove URLs
    - Normalize whitespace
    - Strip leading/trailing spaces
    """
    text = text.lower()
    text = URL_PATTERN.sub("", text)
    text = WHITESPACE_PATTERN.sub(" ", text)
    return text.strip()


# -------------------------------------------------------------------
# Apply cleaning
# -------------------------------------------------------------------
df = df.copy()
df["text"] = df["title"].astype(str).apply(clean_text)

# Drop empty or degenerate text
before = len(df)
df = df[df["text"].str.len() > 0]
after = len(df)

print(f"Removed empty texts: {before - after:,}")

# Drop exact duplicates after cleaning
before = len(df)
df = df.drop_duplicates(subset="text")
after = len(df)

print(f"Removed duplicate texts: {before - after:,}")

# -------------------------------------------------------------------
# Save cleaned dataset
# -------------------------------------------------------------------
df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved cleaned text dataset:")
print(OUTPUT_PATH)
print(f"Final article count: {len(df):,}")

In [ ]:
"""
Text EDA for News Sentiment Analysis
─────────────────────────────────────
Descriptive statistics, distribution analysis, temporal coverage checks,
source diagnostics, and outlier inspection on cleaned news text.

Visual style matches the project-wide light academic palette.
"""

from __future__ import annotations

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde

# =============================================================================
#  GLOBAL STYLE  —  light academic palette
# =============================================================================
BG          = "#FFFFFF"
PANEL_BG    = "#F7F8FA"
BORDER      = "#D0D5DD"
TEXT_PRI    = "#1A1D23"
TEXT_MUT    = "#6B7280"
ACCENT_BLUE = "#2563EB"
ACCENT_RED  = "#DC2626"
ACCENT_AMB  = "#D97706"
ACCENT_GRN  = "#16A34A"
ACCENT_PUR  = "#7C3AED"

plt.rcParams.update({
    "figure.facecolor":   BG,
    "axes.facecolor":     PANEL_BG,
    "axes.edgecolor":     BORDER,
    "axes.labelcolor":    TEXT_PRI,
    "axes.titlecolor":    TEXT_PRI,
    "axes.grid":          True,
    "grid.color":         BORDER,
    "grid.linewidth":     0.55,
    "grid.alpha":         0.7,
    "grid.linestyle":     "--",
    "xtick.color":        TEXT_MUT,
    "ytick.color":        TEXT_MUT,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "axes.titlesize":     12,
    "axes.titleweight":   "bold",
    "axes.titlepad":      10,
    "axes.labelsize":     10,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "font.family":        "DejaVu Sans",
    "text.color":         TEXT_PRI,
    "legend.facecolor":   BG,
    "legend.edgecolor":   BORDER,
    "legend.fontsize":    9,
    "savefig.facecolor":  BG,
    "savefig.dpi":        180,
    "figure.dpi":         110,
})

# =============================================================================
#  SHARED HELPERS
# =============================================================================

# def :
#     fig.text(0.99, 0.005, "Confidential Research", fontsize=7,
#              color=TEXT_MUT, ha="right", va="bottom", alpha=0.5, style="italic")


def _spine_color(ax):
    ax.spines["bottom"].set_color(BORDER)
    ax.spines["left"].set_color(BORDER)


def _stats_badge(ax, data: pd.Series):
    """Monospace stats box in the top-right corner of an axes."""
    txt = (
        f"n = {len(data):,}\n"
        f"μ = {data.mean():.2f}\n"
        f"σ = {data.std():.2f}\n"
        f"Skew = {data.skew():.2f}\n"
        f"Kurt = {data.kurtosis():.2f}"
    )
    ax.text(0.97, 0.97, txt, transform=ax.transAxes,
            ha="right", va="top", fontsize=8.5, color=TEXT_MUT,
            fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.45", facecolor=BG,
                      edgecolor=BORDER, linewidth=0.8, alpha=0.95))


def _gradient_hist_kde(ax, data: pd.Series, color: str,
                       bins: int = 45, xlabel: str = ""):
    """Gradient-bar histogram + KDE overlay, matching project style."""
    counts, edges = np.histogram(data.dropna(), bins=bins)
    norm_h = counts / counts.max()
    centers = 0.5 * (edges[:-1] + edges[1:])

    for x, cnt, nh, lo, hi in zip(centers, counts, norm_h,
                                   edges[:-1], edges[1:]):
        rgba = plt.matplotlib.colors.to_rgba(color, alpha=0.25 + 0.60 * nh)
        ax.bar(x, cnt, width=(hi - lo) * 0.90,
               color=rgba, linewidth=0, zorder=3)

    clean = data.dropna()
    kde    = gaussian_kde(clean, bw_method="scott")
    xr     = np.linspace(clean.min(), clean.max(), 400)
    scale  = counts.max() / kde(xr).max()
    ax.plot(xr, kde(xr) * scale, color=color, linewidth=2.2, zorder=5)
    ax.fill_between(xr, kde(xr) * scale, alpha=0.10, color=color, zorder=4)

    # P5 / P95 / Median reference lines — staggered heights to avoid overlap
    ymax   = counts.max()
    heights = [0.97, 0.78, 0.59]   # three distinct vertical positions
    for (q, label), y_frac in zip(
            [(0.05, "P5"), (0.50, "Median"), (0.95, "P95")], heights):
        val = clean.quantile(q)
        ls  = "--" if q == 0.50 else "-."
        clr = TEXT_PRI if q == 0.50 else ACCENT_AMB
        ax.axvline(val, linestyle=ls, linewidth=1.1, color=clr, alpha=0.8, zorder=6)
        ax.text(val, ymax * y_frac, f" {label}: {val:.0f}",
                fontsize=7.5, color=clr, va="top", rotation=90, alpha=0.90)

    ax.set_xlabel(xlabel, labelpad=5)
    ax.set_ylabel("Frequency", labelpad=5)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
    ax.set_xlim(edges[0], edges[-1])
    ax.set_ylim(0, ymax * 1.22)
    _spine_color(ax)
    _stats_badge(ax, clean)


def savefig(fig, path):
    
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


# =============================================================================
#  PATHS & CONFIG
# =============================================================================
INPUT_PATH    = "../data/processed/sentiment_analysis/news_text_cleaned.csv"
VIS_DIR       = "../data/visuals/sentiment_analysis"
ORIGINAL_COUNT = 42_141          # from prior deduplication step

os.makedirs(VIS_DIR, exist_ok=True)

# =============================================================================
#  LOAD & FEATURE ENGINEERING
# =============================================================================
df = pd.read_csv(INPUT_PATH, parse_dates=["date"])
df["char_len"] = df["text"].str.len()
df["word_len"] = df["text"].str.split().str.len()

print(f"Loaded cleaned articles : {len(df):,}")
print(f"Date range              : {df['date'].min().date()} → {df['date'].max().date()}")

# =============================================================================
#  1. SUMMARY STATISTICS  (console)
# =============================================================================
stats = df["word_len"].describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95])
print(f"\n{'─'*45}")
print("  WORD COUNT STATISTICS")
print(f"{'─'*45}")
print(stats.to_string())

dup_removed_pct = 1 - (len(df) / ORIGINAL_COUNT)
print(f"\nDuplicate removal rate  : {dup_removed_pct:.2%}")
print(f"Articles retained       : {len(df):,} / {ORIGINAL_COUNT:,}")

daily_counts = df.set_index("date").resample("D").size()
print(f"\nDays with zero articles : {(daily_counts == 0).sum()}")

print(f"\n{'─'*45}")
print("  TOP SOURCES (post-cleaning)")
print(f"{'─'*45}")
print(df["domain"].value_counts().head(15).to_string())

print(f"\n{'─'*45}")
print("  SHORTEST HEADLINES")
print(f"{'─'*45}")
print(df.nsmallest(5, "word_len")[["text", "word_len"]].to_string(index=False))

print(f"\n{'─'*45}")
print("  LONGEST HEADLINES")
print(f"{'─'*45}")
print(df.nlargest(5, "word_len")[["text", "word_len"]].to_string(index=False))

# =============================================================================
#  2. TEXT LENGTH DISTRIBUTIONS  —  side-by-side
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5.0))
fig.subplots_adjust(top=0.84, bottom=0.12, left=0.07,
                    right=0.97, wspace=0.30)

_gradient_hist_kde(axes[0], df["word_len"],  ACCENT_BLUE,
                   bins=45, xlabel="Word Count")
axes[0].set_title("Word Count Distribution")

_gradient_hist_kde(axes[1], df["char_len"], ACCENT_GRN,
                   bins=45, xlabel="Character Count")
axes[1].set_title("Character Length Distribution")

fig.suptitle("News Headline Text Length Distributions",
             fontsize=15, fontweight="bold", color=TEXT_PRI, y=0.97)
savefig(fig, f"{VIS_DIR}/text_length_distributions.png")

# =============================================================================
#  3. TEMPORAL COVERAGE
# =============================================================================
daily_counts   = df.set_index("date").resample("D").size()
monthly_counts = df.set_index("date").resample("ME").size()
rolling_7d     = daily_counts.rolling(7).mean()

fig, axes = plt.subplots(2, 1, figsize=(12, 7.5))
fig.subplots_adjust(top=0.88, bottom=0.10, left=0.09,
                    right=0.97, hspace=0.52)

# — Monthly bar chart
ax = axes[0]
bar_colors = [
    ACCENT_RED if v == monthly_counts.min()
    else (ACCENT_GRN if v == monthly_counts.max() else ACCENT_BLUE)
    for v in monthly_counts.values
]
ax.bar(monthly_counts.index, monthly_counts.values,
       width=20, color=bar_colors, alpha=0.82,
       edgecolor="white", linewidth=0.5, zorder=3)
ax.plot(monthly_counts.index, monthly_counts.values,
        color=ACCENT_BLUE, linewidth=1.2, alpha=0.5, zorder=4)

# Annotate min / max
for idx_val, label_clr, va in [
    (monthly_counts.idxmax(), ACCENT_GRN, "bottom"),
    (monthly_counts.idxmin(), ACCENT_RED, "top"),
]:
    v = monthly_counts[idx_val]
    ax.annotate(f"{v:,}", xy=(idx_val, v),
                xytext=(0, 6 if va == "bottom" else -6),
                textcoords="offset points",
                ha="center", va=va, fontsize=8,
                color=ACCENT_GRN if va == "bottom" else ACCENT_RED,
                fontweight="bold")

ax.set_title("Monthly Article Volume")
ax.set_ylabel("Articles")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
_spine_color(ax)

# — Daily counts with 7-day rolling average
ax = axes[1]
ax.fill_between(daily_counts.index, daily_counts.values,
                alpha=0.18, color=ACCENT_GRN, zorder=2)
ax.plot(daily_counts.index, daily_counts.values,
        color=ACCENT_GRN, linewidth=0.7, alpha=0.55, zorder=3)
ax.plot(rolling_7d.index, rolling_7d.values,
        color=ACCENT_AMB, linewidth=1.2, zorder=4, label="7-day MA")

# Highlight zero-article days
zero_days = daily_counts[daily_counts == 0]
if len(zero_days):
    ax.scatter(zero_days.index, zero_days.values,
               color=ACCENT_RED, s=25, zorder=5,
               label=f"Zero-article days ({len(zero_days)})",
               edgecolors="white", linewidths=0.4)

ax.legend(framealpha=0.9)
ax.set_title("Daily Article Volume  (7-Day Rolling Average)")
ax.set_ylabel("Articles / Day")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
_spine_color(ax)

fig.suptitle("Temporal Coverage of News Articles",
             fontsize=15, fontweight="bold", color=TEXT_PRI, y=0.97)
savefig(fig, f"{VIS_DIR}/temporal_coverage.png")

# =============================================================================
#  4. SOURCE DISTRIBUTION
# =============================================================================
top_n   = 15
top_src = df["domain"].value_counts().head(top_n)
total   = len(df)
share   = top_src / total * 100

fig, ax = plt.subplots(figsize=(11, 5.2))
fig.subplots_adjust(top=0.84, bottom=0.22, left=0.08, right=0.97)

# Colour gradient: darker bar = more articles
norm_h   = top_src.values / top_src.max()
bar_rgba = [plt.matplotlib.colors.to_rgba(ACCENT_BLUE, alpha=0.30 + 0.60 * n)
            for n in norm_h]

bars = ax.bar(range(top_n), top_src.values,
              color=bar_rgba, edgecolor="white",
              linewidth=0.6, zorder=3)

# Annotate count + share
for i, (bar, cnt, pct) in enumerate(zip(bars, top_src.values, share.values)):
    ax.text(bar.get_x() + bar.get_width() / 2,
            cnt + top_src.max() * 0.008,
            f"{cnt:,}\n({pct:.1f}%)",
            ha="center", va="bottom",
            fontsize=7.5, color=TEXT_PRI, fontweight="bold")

ax.set_xticks(range(top_n))
ax.set_xticklabels(top_src.index, rotation=40, ha="right", fontsize=9)
ax.set_ylabel("Article Count")
ax.set_ylim(0, top_src.max() * 1.22)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
ax.set_title(f"Top {top_n} News Sources (Post-Cleaning)")
_spine_color(ax)

# Cumulative share badge
cum_share = share.sum()
ax.text(0.97, 0.97,
        f"Top {top_n} sources\ncover {cum_share:.1f}%\nof all articles",
        transform=ax.transAxes, ha="right", va="top",
        fontsize=9, color=TEXT_MUT,
        bbox=dict(boxstyle="round,pad=0.5", facecolor=BG,
                  edgecolor=BORDER, linewidth=0.8))

fig.suptitle("News Source Distribution",
             fontsize=15, fontweight="bold", color=TEXT_PRI, y=0.97)
savefig(fig, f"{VIS_DIR}/top_sources.png")

# =============================================================================
#  5. DUPLICATE / DATA QUALITY DASHBOARD
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5.0))
fig.subplots_adjust(top=0.84, bottom=0.12, left=0.08,
                    right=0.97, wspace=0.32)

# — Retention donut
ax = axes[0]
retained  = len(df)
removed   = ORIGINAL_COUNT - retained
sizes     = [retained, removed]
clrs      = [ACCENT_GRN, ACCENT_RED]
explode   = (0.03, 0.03)
wedges, _ = ax.pie(sizes, colors=clrs, explode=explode,
                   startangle=90, wedgeprops=dict(width=0.55, edgecolor=BG))
ax.text(0, -0.15,
        f"{retained:,}\nretained\n({100 - dup_removed_pct*100:.1f}%)",
        ha="center", va="center", fontsize=11,
        fontweight="bold", color=TEXT_PRI)
ax.legend(["Retained", f"Removed ({dup_removed_pct:.1%})"],
          loc="upper center", bbox_to_anchor=(0.5, -0.06),
          framealpha=0.9, fontsize=9, ncol=2)
ax.set_title("Deduplication Retention")
ax.set_facecolor(BG)

# — Word-length bucket breakdown
ax = axes[1]
_last_edge = max(int(df["word_len"].max()) + 1, 101)
buckets = pd.cut(df["word_len"],
                 bins=[0, 5, 10, 20, 50, 100, _last_edge],
                 labels=["≤5", "6–10", "11–20", "21–50", "51–100", ">100"])
bucket_counts = buckets.value_counts().sort_index()
bucket_pct    = bucket_counts / bucket_counts.sum() * 100
bucket_colors = [ACCENT_BLUE, ACCENT_GRN, ACCENT_AMB,
                 ACCENT_PUR, ACCENT_RED, TEXT_MUT]

bars = ax.bar(range(len(bucket_counts)), bucket_counts.values,
              color=bucket_colors, alpha=0.82,
              edgecolor="white", linewidth=0.6, zorder=3)
for bar, cnt, pct in zip(bars, bucket_counts.values, bucket_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            cnt + bucket_counts.max() * 0.012,
            f"{cnt:,}\n({pct:.1f}%)",
            ha="center", va="bottom",
            fontsize=8.5, color=TEXT_PRI, fontweight="bold")

ax.set_xticks(range(len(bucket_counts)))
ax.set_xticklabels(bucket_counts.index, fontsize=10)
ax.set_xlabel("Word Count Bucket")
ax.set_ylabel("Number of Articles")
ax.set_ylim(0, bucket_counts.max() * 1.25)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
ax.set_title("Article Length Bucket Breakdown")
_spine_color(ax)

fig.suptitle("Data Quality & Coverage Summary",
             fontsize=15, fontweight="bold", color=TEXT_PRI, y=0.97)
savefig(fig, f"{VIS_DIR}/data_quality_dashboard.png")

# =============================================================================
#  DONE
# =============================================================================
print(f"\n{'─'*45}")
print("  EDA complete.  Plots saved to:")
print(f"  {VIS_DIR}")
print(f"{'─'*45}")

In [ ]:
"""
Token Length Truncation Analysis for FinBERT

This cell:
- Loads the FinBERT tokenizer
- Computes true token lengths without truncation
- Evaluates truncation impact for common max_length values
- Summarises results in a clear, reproducible table
"""

from transformers import AutoTokenizer
import numpy as np
import pandas as pd

# -------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------
INPUT_PATH = "../data/processed/sentiment_analysis/news_text_cleaned.csv"

# -------------------------------------------------------------------
# Load data
# -------------------------------------------------------------------
df = pd.read_csv(INPUT_PATH)

texts = df["text"].tolist()
total_samples = len(texts)

print(f"Analysing token lengths for {total_samples:,} headlines")

# -------------------------------------------------------------------
# Load FinBERT tokenizer
# -------------------------------------------------------------------
MODEL_CHECKPOINT = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)


# -------------------------------------------------------------------
# Compute true token lengths (no truncation)
# -------------------------------------------------------------------
df = df[df["text"].notna()]
df["text"] = df["text"].astype(str)

texts = df["text"].tolist()

encodings = tokenizer(
    texts,
    add_special_tokens=True,
    truncation=False,
    padding=False
)

token_lengths = np.array([len(ids) for ids in encodings["input_ids"]])

# -------------------------------------------------------------------
# Truncation analysis
# -------------------------------------------------------------------
thresholds = [16, 24, 32, 48, 64]

results = []

for max_len in thresholds:
    truncated = np.sum(token_lengths > max_len)
    results.append({
        "Max tokens": max_len,
        "Articles truncated": truncated,
        "Percent truncated": truncated / total_samples
    })

truncation_df = pd.DataFrame(results)

truncation_df["Percent truncated"] = truncation_df["Percent truncated"].map(
    lambda x: f"{x:.2%}"
)

# -------------------------------------------------------------------
# Summary statistics (context)
# -------------------------------------------------------------------
length_stats = {
    "Mean tokens": token_lengths.mean(),
    "Median tokens": np.median(token_lengths),
    "95th percentile": np.percentile(token_lengths, 95),
    "Max tokens": token_lengths.max(),
}

print("\nToken length summary:")
for k, v in length_stats.items():
    print(f"{k:<18}: {v:.2f}")

print("\nTruncation impact by max_length:")
truncation_df

In [ ]:
"""
Article-Level Tokenisation for FinBERT Inference

Tokenises cleaned news headlines using the FinBERT tokenizer with
empirically chosen truncation length. Outputs tokenised tensors
ready for batched sentiment inference.
"""

from __future__ import annotations

import pandas as pd
import torch
from transformers import AutoTokenizer

# -------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------
INPUT_PATH = "../data/processed/sentiment_analysis/news_text_cleaned.csv"

# -------------------------------------------------------------------
# Parameters
# -------------------------------------------------------------------
MODEL_CHECKPOINT = "ProsusAI/finbert"
MAX_LENGTH = 32
BATCH_SIZE = 64

# -------------------------------------------------------------------
# Load data
# -------------------------------------------------------------------
df = pd.read_csv(INPUT_PATH)

texts = df["text"].astype(str).tolist()
num_samples = len(texts)

print(f"Tokenising {num_samples:,} headlines")

# -------------------------------------------------------------------
# Load tokenizer
# -------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# -------------------------------------------------------------------
# Tokenisation (batched, dynamic padding)
# -------------------------------------------------------------------
def tokenize_batch(text_batch: list[str]) -> dict[str, torch.Tensor]:
    """
    Tokenise a batch of text inputs for FinBERT inference.

    Args:
        text_batch (list[str]): List of headline texts.

    Returns:
        dict[str, torch.Tensor]: Tokenised inputs (input_ids, attention_mask).
    """
    return tokenizer(
        text_batch,
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )


# Example batch (sanity check)
sample_batch = texts[:BATCH_SIZE]
encoded = tokenize_batch(sample_batch)

print("\nTokenised batch shapes:")
for k, v in encoded.items():
    print(f"{k:<15}: {tuple(v.shape)}")

print("\nTokenisation ready for inference.")

In [ ]:
"""
Baseline FinBERT Sentiment Inference (Article-Level)

Runs pretrained FinBERT on cleaned news headlines to obtain
article-level sentiment probabilities without any fine-tuning.
"""

from __future__ import annotations

import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader

# -------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------
INPUT_PATH = "../data/processed/sentiment_analysis/news_text_cleaned.csv"
OUTPUT_PATH = "../data/processed/sentiment_analysis/news_with_sentiment.csv"

# -------------------------------------------------------------------
# Parameters
# -------------------------------------------------------------------
MODEL_CHECKPOINT = "ProsusAI/finbert"
MAX_LENGTH = 32
BATCH_SIZE = 64

# -------------------------------------------------------------------
# Device
# -------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -------------------------------------------------------------------
# Load data
# -------------------------------------------------------------------
df = pd.read_csv(INPUT_PATH)
texts = df["text"].astype(str).tolist()

print(f"Running inference on {len(texts):,} articles")

# -------------------------------------------------------------------
# Load tokenizer and model
# -------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT)
model.to(device)
model.eval()

# -------------------------------------------------------------------
# Tokenisation helper
# -------------------------------------------------------------------
def tokenize_batch(text_batch: list[str]) -> dict[str, torch.Tensor]:
    return tokenizer(
        text_batch,
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

# -------------------------------------------------------------------
# DataLoader (index-based for alignment safety)
# -------------------------------------------------------------------
indices = list(range(len(texts)))
loader = DataLoader(indices, batch_size=BATCH_SIZE, shuffle=False)

# -------------------------------------------------------------------
# Inference loop
# -------------------------------------------------------------------
all_probs = []

with torch.no_grad():
    for batch_idx in loader:
        batch_texts = [texts[i] for i in batch_idx]

        enc = tokenize_batch(batch_texts)
        enc = {k: v.to(device) for k, v in enc.items()}

        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=1)

        all_probs.append(probs.cpu().numpy())

# -------------------------------------------------------------------
# Collect results
# -------------------------------------------------------------------
probs = np.vstack(all_probs)

sentiment_df = pd.DataFrame(
    probs,
    columns=["prob_negative", "prob_neutral", "prob_positive"]
)

# -------------------------------------------------------------------
# Attach to original dataframe
# -------------------------------------------------------------------
df = df.reset_index(drop=True)
df = pd.concat([df, sentiment_df], axis=1)

# Optional scalar sentiment score (continuous)
df["sentiment_score"] = df["prob_positive"] - df["prob_negative"]

# -------------------------------------------------------------------
# Save results
# -------------------------------------------------------------------
df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved article-level sentiment dataset:")
print(OUTPUT_PATH)
print("Inference complete.")

In [ ]:
df = pd.read_csv("../data/processed/sentiment_analysis/news_with_sentiment.csv")

df[[
    "prob_negative",
    "prob_neutral",
    "prob_positive",
    "sentiment_score"
]].describe()

##### The pretrained FinBERT model produces stable and well-distributed sentiment probabilities across the corpus, with neutral sentiment most prevalent and a wide dispersion in sentiment scores suitable for downstream aggregation and stress detection.

In [ ]:
"""
Daily Aggregation of Article-Level Sentiment (Equal-Weighted)
Leakage-safe: sentiment is shifted forward by 1 day.
"""

from __future__ import annotations
import pandas as pd

# -------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------

INPUT_PATH  = "../data/processed/sentiment_analysis/news_with_sentiment.csv"
OUTPUT_PATH = "../data/processed/sentiment_analysis/daily_sentiment_features.csv"

# -------------------------------------------------------------------
# Keyword group labels
# -------------------------------------------------------------------

CALM_GROUPS   = {1, 2, 3}
STRESS_GROUPS = {4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14}
NEUTRAL_GROUPS = {15, 16, 17}

# -------------------------------------------------------------------
# Load data
# -------------------------------------------------------------------

df = pd.read_csv(INPUT_PATH, parse_dates=["date"])
print(f"Loaded article-level sentiment data: {len(df):,} rows")

# -------------------------------------------------------------------
# Ensure required columns exist
# -------------------------------------------------------------------

required_cols = ["date", "sentiment_score", "prob_negative", "prob_neutral", "prob_positive", "kw_group"]
missing = set(required_cols) - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# -------------------------------------------------------------------
# Daily aggregation
# -------------------------------------------------------------------

daily = (
    df
    .groupby(df["date"].dt.date)
    .agg(
        sentiment_mean       = ("sentiment_score", "mean"),
        sentiment_median     = ("sentiment_score", "median"),
        sentiment_std        = ("sentiment_score", "std"),
        sentiment_min        = ("sentiment_score", "min"),
        pct_negative         = ("prob_negative", lambda x: (x > 0.5).mean()),
        article_count        = ("sentiment_score", "size"),
        stress_article_count = ("kw_group", lambda x: x.isin(STRESS_GROUPS).sum()),
        calm_article_count   = ("kw_group", lambda x: x.isin(CALM_GROUPS).sum()),
    )
    .reset_index()
    .rename(columns={"date": "date"})
)

daily["date"] = pd.to_datetime(daily["date"])

# Stress ratio: proportion of articles from stress keyword groups
daily["stress_ratio"] = daily["stress_article_count"] / daily["article_count"].replace(0, 1)

# -------------------------------------------------------------------
# Shift forward by 1 day (leakage-safe)
# -------------------------------------------------------------------

daily["date"] = daily["date"] + pd.Timedelta(days=1)

print(f"Generated leakage-safe daily sentiment features: {len(daily):,} days")

# -------------------------------------------------------------------
# Save
# -------------------------------------------------------------------

daily.to_csv(OUTPUT_PATH, index=False)
print("\nSaved daily sentiment features:")
print(OUTPUT_PATH)

##### For report:
##### Market stress is treated as a contemporaneous classification problem, where daily market conditions are classified as stressed or normal using information available up to that day. This framing aligns with financial stability monitoring practices and enables direct comparison with traditional volatility-based methods.

In [ ]:
"""
Sentiment Feature Engineering Debug
"""

from __future__ import annotations

import pandas as pd

# -------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------
INPUT_PATH = "../data/processed/sentiment_analysis/daily_sentiment_features.csv"
OUTPUT_PATH = "../data/processed/sentiment_analysis/daily_sentiment_engineered_debug.csv"

# -------------------------------------------------------------------
# Parameters
# -------------------------------------------------------------------
ROLLING_WINDOWS = [5, 10]

# -------------------------------------------------------------------
# Load data
# -------------------------------------------------------------------
df = pd.read_csv(INPUT_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Loaded daily sentiment data: {len(df):,} days")

# -------------------------------------------------------------------
# Core time-series features
# -------------------------------------------------------------------
# Day-over-day change
df["sentiment_change"] = df["sentiment_mean"].diff()

# Rolling features
for w in ROLLING_WINDOWS:
    df[f"sentiment_mean_roll_{w}"] = df["sentiment_mean"].rolling(window=w, min_periods=1).mean()
    df[f"sentiment_std_roll_{w}"]  = df["sentiment_mean"].rolling(window=w, min_periods=1).std()
    df[f"stress_ratio_roll_{w}"]   = df["stress_ratio"].rolling(window=w, min_periods=1).mean()

df["stress_ratio_change"] = df["stress_ratio"].diff()

# -------------------------------------------------------------------
# Optional normalization (useful for ML stability)
# -------------------------------------------------------------------
df["sentiment_zscore_10"] = (
    (df["sentiment_mean"] - df["sentiment_mean"].rolling(10).mean())
    / df["sentiment_mean"].rolling(10).std()
)

# -------------------------------------------------------------------
# Save engineered features
# -------------------------------------------------------------------
df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved engineered daily sentiment features:")
print(OUTPUT_PATH)

print("\nGenerated features:")
engineered_cols = [c for c in df.columns if c not in ["date"]]
for c in engineered_cols:
    print(f" - {c}")

In [ ]:
"""
Sentiment Feature Engineering (Time-Series Dynamics)

Creates rolling and change-based sentiment features from
daily aggregated sentiment signals.
"""

from __future__ import annotations

import pandas as pd

# -------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------
INPUT_PATH = "../data/processed/sentiment_analysis/daily_sentiment_features.csv"
OUTPUT_PATH = "../data/processed/sentiment_analysis/daily_sentiment_engineered.csv"

# -------------------------------------------------------------------
# Parameters
# -------------------------------------------------------------------
ROLLING_WINDOWS = [5, 10]

# -------------------------------------------------------------------
# Load data
# -------------------------------------------------------------------
df = pd.read_csv(INPUT_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Loaded daily sentiment data: {len(df):,} days")

# -------------------------------------------------------------------
# Save engineered features
# -------------------------------------------------------------------
df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved engineered daily sentiment features:")
print(OUTPUT_PATH)

print("\nGenerated features:")
engineered_cols = [c for c in df.columns if c not in ["date"]]
for c in engineered_cols:
    print(f" - {c}")

##### Rolling and change-based sentiment features are constructed to capture persistence, dispersion, and shocks in the information environment, as market stress is driven not only by sentiment levels but also by uncertainty and abrupt shifts in news tone.

In [ ]:
"""
Coverage & Sanity Checks for Daily Sentiment Features
─────────────────────────────────────────────────────
Visual style matches the project-wide light academic palette.
"""

from __future__ import annotations

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from scipy.stats import gaussian_kde

# =============================================================================
#  GLOBAL STYLE  —  light academic palette
# =============================================================================
BG          = "#FFFFFF"
PANEL_BG    = "#F7F8FA"
BORDER      = "#D0D5DD"
TEXT_PRI    = "#1A1D23"
TEXT_MUT    = "#6B7280"
ACCENT_BLUE = "#2563EB"
ACCENT_RED  = "#DC2626"
ACCENT_AMB  = "#D97706"
ACCENT_GRN  = "#16A34A"
ACCENT_PUR  = "#7C3AED"

plt.rcParams.update({
    "figure.facecolor":   BG,
    "axes.facecolor":     PANEL_BG,
    "axes.edgecolor":     BORDER,
    "axes.labelcolor":    TEXT_PRI,
    "axes.titlecolor":    TEXT_PRI,
    "axes.grid":          True,
    "grid.color":         BORDER,
    "grid.linewidth":     0.55,
    "grid.alpha":         0.7,
    "grid.linestyle":     "--",
    "xtick.color":        TEXT_MUT,
    "ytick.color":        TEXT_MUT,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "axes.titlesize":     12,
    "axes.titleweight":   "bold",
    "axes.titlepad":      10,
    "axes.labelsize":     10,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "font.family":        "DejaVu Sans",
    "text.color":         TEXT_PRI,
    "legend.facecolor":   BG,
    "legend.edgecolor":   BORDER,
    "legend.fontsize":    9,
    "savefig.facecolor":  BG,
    "savefig.dpi":        180,
    "figure.dpi":         110,
})

VIS_DIR = "../data/visuals/sentiment_analysis"
os.makedirs(VIS_DIR, exist_ok=True)

# =============================================================================
#  HELPERS
# =============================================================================

# def :
#     fig.text(0.99, 0.005, "Confidential Research", fontsize=7,
#              color=TEXT_MUT, ha="right", va="bottom", alpha=0.5, style="italic")


def _spine(ax):
    ax.spines["bottom"].set_color(BORDER)
    ax.spines["left"].set_color(BORDER)


def _date_axis(ax, interval=6):
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=interval))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")


def _stats_badge(ax, series: pd.Series, extra: dict | None = None):
    lines = [
        f"n     = {series.notna().sum():,}",
        f"μ     = {series.mean():.4f}",
        f"σ     = {series.std():.4f}",
        f"P1    = {series.quantile(0.01):.4f}",
        f"P99   = {series.quantile(0.99):.4f}",
        f"Skew  = {series.skew():.2f}",
    ]
    if extra:
        for k, v in extra.items():
            lines.append(f"{k:<6} = {v}")
    ax.text(0.97, 0.97, "\n".join(lines),
            transform=ax.transAxes, ha="right", va="top",
            fontsize=8, color=TEXT_MUT, fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.48", facecolor=BG,
                      edgecolor=BORDER, linewidth=0.8, alpha=0.95))


def _crisis_band(ax, start, end, label, color=ACCENT_RED, alpha=0.10, y_frac=0.97):
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
               color=color, alpha=alpha, zorder=1)
    mid = pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start)) / 2
    ylim = ax.get_ylim()
    ax.text(mid, ylim[0] + (ylim[1] - ylim[0]) * y_frac, label,
            ha="center", va="top", fontsize=7.5,
            color=color, style="italic", alpha=0.85)


def savefig(fig, path):
    
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


CRISIS_PERIODS = [
    ("COVID-19",       "2020-02-15", "2020-04-30", ACCENT_RED),
    ("Inflation shock","2022-01-01", "2022-10-31", ACCENT_AMB),
]

# =============================================================================
#  LOAD DATA
# =============================================================================
INPUT_PATH = "../data/processed/sentiment_analysis/daily_sentiment_engineered_debug.csv"

df = pd.read_csv(INPUT_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

full_range   = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
missing_days = full_range.difference(df["date"])

print(f"Loaded engineered sentiment data : {len(df):,} days")
print(f"Calendar days covered            : {len(full_range):,}")
print(f"Missing calendar days            : {len(missing_days):,}")

miss_ratio = df.isna().mean()
miss_ratio = miss_ratio[miss_ratio > 0]
if miss_ratio.empty:
    print("\nNo missing values detected.")
else:
    print("\nMissing value ratio by feature:")
    print(miss_ratio.to_string())

for col in ["sentiment_mean", "sentiment_zscore_10", "stress_ratio"]:
    print(f"\n{col} extremes:")
    print(df[col].describe(percentiles=[0.01, 0.99]).to_string())

print("\nCrisis-period snapshots:")
for label, start, end, _ in CRISIS_PERIODS:
    sub = df[(df["date"] >= start) & (df["date"] <= end)]
    print(f"\n  {label} ({start} → {end}):")
    print(sub["sentiment_mean"].describe().to_string())
    print(f"  Stress ratio mean : {sub['stress_ratio'].mean():.3f}")

# =============================================================================
#  PLOT 1 — SENTIMENT MEAN TIME SERIES  (with crisis bands + 20-day MA)
# =============================================================================
fig, ax = plt.subplots(figsize=(12, 4.6))
fig.subplots_adjust(top=0.84, bottom=0.14, left=0.08, right=0.97)

ma20 = df["sentiment_mean"].rolling(20).mean()
zero_line = 0

# Colour-split fills: positive sentiment green, negative red
pos = df["sentiment_mean"].clip(lower=zero_line)
neg = df["sentiment_mean"].clip(upper=zero_line)
ax.fill_between(df["date"], pos, zero_line, alpha=0.30,
                color=ACCENT_GRN, zorder=2, label="Positive")
ax.fill_between(df["date"], neg, zero_line, alpha=0.30,
                color=ACCENT_RED, zorder=2, label="Negative")
ax.plot(df["date"], df["sentiment_mean"],
        color=TEXT_MUT, linewidth=0.7, alpha=0.55, zorder=3)
ax.plot(df["date"], ma20,
        color=ACCENT_BLUE, linewidth=2.0, zorder=4, label="20-day MA")
ax.axhline(0, color=BORDER, linewidth=0.9, zorder=3)

for label, start, end, color in CRISIS_PERIODS:
    _crisis_band(ax, start, end, label, color=color)

ax.legend(framealpha=0.9, loc="upper right")
ax.set_ylabel("Sentiment Mean")
ax.set_title("Daily Sentiment Mean Over Time")
_date_axis(ax)
_spine(ax)
_stats_badge(ax, df["sentiment_mean"])

fig.suptitle("Sentiment Feature Coverage & Sanity Checks",
             fontsize=15, fontweight="bold", color=TEXT_PRI, y=0.97)
savefig(fig, f"{VIS_DIR}/sentiment_mean_timeseries.png")

# =============================================================================
#  PLOT 2 — ARTICLE VOLUME  (daily bars + 7-day MA + missing-day markers)
# =============================================================================
fig, ax = plt.subplots(figsize=(12, 4.2))
fig.subplots_adjust(top=0.84, bottom=0.14, left=0.08, right=0.97)

ax.bar(df["date"], df["article_count"],
       color=ACCENT_BLUE, alpha=0.35, width=1, zorder=2)
ma7 = df["article_count"].rolling(7).mean()
ax.plot(df["date"], ma7,
        color=ACCENT_BLUE, linewidth=1.8, zorder=3, label="7-day MA")

# Mark zero / missing volume days
zero_vol = df[df["article_count"] == 0]
if len(zero_vol):
    ax.scatter(zero_vol["date"], zero_vol["article_count"],
               color=ACCENT_RED, s=28, zorder=5,
               label=f"Zero-article days ({len(zero_vol)})",
               edgecolors="white", linewidths=0.4)

# Coverage badge
ax.text(0.02, 0.97,
        f"Missing calendar days: {len(missing_days):,}\n"
        f"Date range: {df['date'].min().date()} → {df['date'].max().date()}",
        transform=ax.transAxes, ha="left", va="top",
        fontsize=8.5, color=TEXT_MUT,
        bbox=dict(boxstyle="round,pad=0.45", facecolor=BG,
                  edgecolor=BORDER, linewidth=0.8))

ax.legend(framealpha=0.9)
ax.set_ylabel("Article Count")
ax.set_title("Daily News Volume Over Time")
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
_date_axis(ax)
_spine(ax)

fig.suptitle("Sentiment Feature Coverage & Sanity Checks",
             fontsize=15, fontweight="bold", color=TEXT_PRI, y=0.97)
savefig(fig, f"{VIS_DIR}/article_volume_timeseries.png")

# =============================================================================
#  PLOT 3 — STRESS RATIO TIME SERIES  (with regime threshold + crisis bands)
# =============================================================================
fig, ax = plt.subplots(figsize=(12, 4.2))
fig.subplots_adjust(top=0.84, bottom=0.14, left=0.08, right=0.97)

STRESS_THRESHOLD = df["stress_ratio"].quantile(0.75)

ax.fill_between(df["date"], df["stress_ratio"],
                alpha=0.18, color=ACCENT_PUR, zorder=2)
ax.plot(df["date"], df["stress_ratio"],
        color=ACCENT_PUR, linewidth=1.3, zorder=3)

# Threshold band
ax.axhline(STRESS_THRESHOLD, linestyle="--", linewidth=1.1,
           color=ACCENT_AMB, alpha=0.80, zorder=4)
ax.text(df["date"].iloc[-1], STRESS_THRESHOLD,
        f"  P75 = {STRESS_THRESHOLD:.3f}",
        fontsize=8, color=ACCENT_AMB, va="bottom")

# Highlight elevated stress periods
elevated = df["stress_ratio"] >= STRESS_THRESHOLD
ax.fill_between(df["date"], df["stress_ratio"],
                where=elevated, alpha=0.30,
                color=ACCENT_RED, zorder=3, label="Elevated stress (≥P75)")

for label, start, end, color in CRISIS_PERIODS:
    _crisis_band(ax, start, end, label, color=color)

ax.legend(framealpha=0.9)
ax.set_ylabel("Stress Ratio")
ax.set_title("Daily Stress Ratio Over Time")
_date_axis(ax)
_spine(ax)
_stats_badge(ax, df["stress_ratio"],
             extra={"P75": f"{STRESS_THRESHOLD:.4f}"})

fig.suptitle("Sentiment Feature Coverage & Sanity Checks",
             fontsize=15, fontweight="bold", color=TEXT_PRI, y=0.97)
savefig(fig, f"{VIS_DIR}/stress_ratio_timeseries.png")

# =============================================================================
#  PLOT 4 — DISTRIBUTION PANEL  (sentiment_mean · zscore · stress_ratio)
# =============================================================================
DIST_COLS = [
    ("sentiment_mean",       "Sentiment Mean",         ACCENT_BLUE),
    ("sentiment_zscore_10",  "Sentiment Z-Score (10d)", ACCENT_PUR),
    ("stress_ratio",         "Stress Ratio",            ACCENT_RED),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5.0))
fig.subplots_adjust(top=0.84, bottom=0.12, left=0.06,
                    right=0.97, wspace=0.30)

for ax, (col, title, color) in zip(axes, DIST_COLS):
    data = df[col].dropna()
    counts, edges = np.histogram(data, bins=45)
    norm_h = counts / counts.max()
    centers = 0.5 * (edges[:-1] + edges[1:])

    # Gradient bars
    for x, cnt, nh, lo, hi in zip(centers, counts, norm_h,
                                   edges[:-1], edges[1:]):
        rgba = plt.matplotlib.colors.to_rgba(color, alpha=0.25 + 0.60 * nh)
        ax.bar(x, cnt, width=(hi - lo) * 0.90,
               color=rgba, linewidth=0, zorder=3)

    # KDE + fill
    kde    = gaussian_kde(data, bw_method="scott")
    xr     = np.linspace(data.min(), data.max(), 400)
    scale  = counts.max() / kde(xr).max()
    ax.plot(xr, kde(xr) * scale, color=color, linewidth=2.2, zorder=5)
    ax.fill_between(xr, kde(xr) * scale,
                    alpha=0.10, color=color, zorder=4)

    # Reference lines
    ymax = counts.max()
    for q, lbl, ls, clr in [
        (0.50, "Median", "--", TEXT_PRI),
        (0.95, "P95",    "-.", ACCENT_AMB),
    ]:
        val = data.quantile(q)
        ax.axvline(val, linestyle=ls, linewidth=1.1,
                   color=clr, alpha=0.80, zorder=6)
        ax.text(val, ymax * 0.97, f" {lbl}\n {val:.3g}",
                fontsize=7.5, color=clr, va="top", rotation=90, alpha=0.90)

    ax.set_title(title)
    ax.set_xlabel(title, labelpad=5)
    ax.set_ylabel("Frequency", labelpad=5)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
    ax.set_xlim(edges[0], edges[-1])
    ax.set_ylim(0, ymax * 1.22)
    _spine(ax)
    _stats_badge(ax, data)

fig.suptitle("Sentiment Feature Distributions",
             fontsize=15, fontweight="bold", color=TEXT_PRI, y=0.97)
savefig(fig, f"{VIS_DIR}/sentiment_feature_distributions.png")

# =============================================================================
#  PLOT 5 — CRISIS-PERIOD COMPARISON  (grouped box/strip summary)
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
fig.subplots_adjust(top=0.84, bottom=0.12, left=0.08,
                    right=0.97, wspace=0.30)

COMPARE_COLS = [
    ("sentiment_mean", "Sentiment Mean",  ACCENT_BLUE),
    ("stress_ratio",   "Stress Ratio",    ACCENT_RED),
]

for ax, (col, ylabel, color) in zip(axes, COMPARE_COLS):
    segments = {"Full\nSample": df[col].dropna()}
    for label, start, end, _ in CRISIS_PERIODS:
        sub = df[(df["date"] >= start) & (df["date"] <= end)][col].dropna()
        segments[label.replace(" ", "\n")] = sub

    positions  = list(range(len(segments)))
    seg_colors = [color, ACCENT_RED, ACCENT_AMB]

    for pos, (seg_label, seg_data), seg_col in zip(
            positions, segments.items(), seg_colors):
        # Thin strip of raw points
        jitter = np.random.default_rng(42).uniform(-0.18, 0.18, len(seg_data))
        ax.scatter(np.full(len(seg_data), pos) + jitter, seg_data,
                   color=seg_col, alpha=0.12, s=6, zorder=2)

        # Box (IQR)
        q1, med, q3 = (seg_data.quantile(0.25),
                       seg_data.median(),
                       seg_data.quantile(0.75))
        iqr = q3 - q1
        box = mpatches.FancyBboxPatch(
            (pos - 0.22, q1), 0.44, iqr,
            boxstyle="round,pad=0.01",
            facecolor=seg_col, alpha=0.25,
            edgecolor=seg_col, linewidth=1.2, zorder=3)
        ax.add_patch(box)

        # Median line
        ax.hlines(med, pos - 0.22, pos + 0.22,
                  color=seg_col, linewidth=2.2, zorder=4)

        # Whiskers (1.5×IQR)
        lo_w = max(seg_data.min(), q1 - 1.5 * iqr)
        hi_w = min(seg_data.max(), q3 + 1.5 * iqr)
        ax.vlines(pos, lo_w, q1, color=seg_col, linewidth=1.0,
                  linestyle="--", alpha=0.6, zorder=3)
        ax.vlines(pos, q3, hi_w, color=seg_col, linewidth=1.0,
                  linestyle="--", alpha=0.6, zorder=3)

        # Median annotation
        ax.text(pos, med, f" {med:.3f}",
                fontsize=8, color="black", va="bottom", fontweight="bold")

    ax.set_xticks(positions)
    ax.set_xticklabels(list(segments.keys()), fontsize=10)
    ax.set_ylabel(ylabel, labelpad=6)
    ax.set_title(f"{ylabel} - Period Comparison")
    _spine(ax)

fig.suptitle("Crisis-Period Feature Comparison",
             fontsize=15, fontweight="bold", color=TEXT_PRI, y=0.97)
savefig(fig, f"{VIS_DIR}/crisis_period_comparison.png")

# =============================================================================
#  DONE
# =============================================================================
print(f"\n{'─'*50}")
print("  Sanity checks complete.  Plots saved to:")
print(f"  {VIS_DIR}")
print(f"{'─'*50}")